# Set Up

In [8]:
import pandas as pd

df = pd.read_csv('../dashboard/top_players_games.csv')
df.head()

,game_id,time_class,opening_name,total_turns,white_username,white_rating,white_result,black_username,black_rating,black_result
0,8efbd65f-e7f8-11f0-8ffe-11616601000f,blitz,Benoni Defense Modern Classical New York Varia...,55,0blivi0usspy,2921,win,Immortal_legend710,2830,timeout
1,a1d6f606-e812-11f0-9cfc-7c4ed201000f,blitz,Queens Gambit Declined Harrwitz Two Knights De...,70,TheFinal_Move,2944,win,0blivi0usspy,2913,timeout
2,a8a6abea-e823-11f0-8085-d502c301000f,blitz,Owens Defense...4.Qe2 e6 5.Nf3 d5 6.e5,29,0blivi0usspy,2919,win,bach12345_lfay,2825,resigned
3,4bdff3c4-e824-11f0-8ffe-11616601000f,blitz,Vienna Game Max Lange Vienna Gambit 3...exf4,39,bach12345_lfay,2827,repetition,0blivi0usspy,2917,repetition
4,8c42d8be-e889-11f0-9c69-fb411201000f,blitz,Queens Gambit Declined Exchange Positional Lin...,34,0blivi0usspy,2924,win,Thecarefreeangel,2876,resigned


In [9]:
df.describe()

,total_turns,white_rating,black_rating
count,323356.000000,323356.000000,323356.000000
mean,43.157319,2894.304806,2894.550400
std,19.855169,270.050576,265.737441
min,0.000000,100.000000,100.000000
25%,30.000000,2809.000000,2808.000000
50%,41.000000,2929.000000,2929.000000
75%,56.000000,3042.000000,3041.000000
max,311.000000,3662.000000,3658.000000


# Feature Engineering

In [ ]:
def determine_winner(row):
    if row['white_result'] == 'win':
        return 'White'
    elif row['black_result'] == 'win':
        return 'Black'
    else:
        return 'Draw'  

def determine_result_type(row):
    if row['white_result'] != 'win':
        return row['white_result']
    else:
        return row['black_result']

df['winner'] = df.apply(determine_winner, axis=1)
df['result'] = df.apply(determine_result_type, axis=1)

df.drop(columns=['white_result', 'black_result'], inplace=True)

df.head()

,game_id,time_class,opening_name,total_turns,white_username,white_rating,black_username,black_rating,winner,result
0,8efbd65f-e7f8-11f0-8ffe-11616601000f,blitz,Benoni Defense Modern Classical New York Varia...,55,0blivi0usspy,2921,Immortal_legend710,2830,White,timeout
1,a1d6f606-e812-11f0-9cfc-7c4ed201000f,blitz,Queens Gambit Declined Harrwitz Two Knights De...,70,TheFinal_Move,2944,0blivi0usspy,2913,White,timeout
2,a8a6abea-e823-11f0-8085-d502c301000f,blitz,Owens Defense...4.Qe2 e6 5.Nf3 d5 6.e5,29,0blivi0usspy,2919,bach12345_lfay,2825,White,resigned
3,4bdff3c4-e824-11f0-8ffe-11616601000f,blitz,Vienna Game Max Lange Vienna Gambit 3...exf4,39,bach12345_lfay,2827,0blivi0usspy,2917,Draw,repetition
4,8c42d8be-e889-11f0-9c69-fb411201000f,blitz,Queens Gambit Declined Exchange Positional Lin...,34,0blivi0usspy,2924,Thecarefreeangel,2876,White,resigned


In [11]:
df['rating_diff'] = df['white_rating'] - df['black_rating']

def check_upset(row):
    if row['winner'] == 'White' and row['white_rating'] < row['black_rating']:
        return True
    elif row['winner'] == 'Black' and row['black_rating'] < row['white_rating']:
        return True
    return False

df['is_upset'] = df.apply(check_upset, axis=1)

def classify_phase(turns):
    if turns <= 20:
        return 'Early'
    elif 21 <= turns <= 40:
        return 'Middlegame'
    else:
        return 'Endgame'

df['game_phase'] = df['total_turns'].apply(classify_phase)

df.head()

,game_id,time_class,opening_name,total_turns,white_username,white_rating,black_username,black_rating,winner,result,rating_diff,is_upset,game_phase
0,8efbd65f-e7f8-11f0-8ffe-11616601000f,blitz,Benoni Defense Modern Classical New York Varia...,55,0blivi0usspy,2921,Immortal_legend710,2830,White,timeout,91,False,Endgame
1,a1d6f606-e812-11f0-9cfc-7c4ed201000f,blitz,Queens Gambit Declined Harrwitz Two Knights De...,70,TheFinal_Move,2944,0blivi0usspy,2913,White,timeout,31,False,Endgame
2,a8a6abea-e823-11f0-8085-d502c301000f,blitz,Owens Defense...4.Qe2 e6 5.Nf3 d5 6.e5,29,0blivi0usspy,2919,bach12345_lfay,2825,White,resigned,94,False,Middlegame
3,4bdff3c4-e824-11f0-8ffe-11616601000f,blitz,Vienna Game Max Lange Vienna Gambit 3...exf4,39,bach12345_lfay,2827,0blivi0usspy,2917,Draw,repetition,-90,False,Middlegame
4,8c42d8be-e889-11f0-9c69-fb411201000f,blitz,Queens Gambit Declined Exchange Positional Lin...,34,0blivi0usspy,2924,Thecarefreeangel,2876,White,resigned,48,False,Middlegame
